In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/atosss/vit5_atoss_results-4.jsonl
/kaggle/input/variation-atoss/vit5_atoss_results-5.jsonl


In [ ]:
import os
import sys
import json
import ast
import re
import torch
import gc
import warnings
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM
from tqdm import tqdm
from peft import PeftModel
from difflib import SequenceMatcher

warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_ID = 'vohuutridung/vit5-large-absa'
eval_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
eval_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map='auto',
)

eval_model.eval()

In [6]:
GEN_CACHE = {}

In [7]:
CATEGORIES = {
    "TỔNG_QUAN","PIN","HIỆU_NĂNG","MÁY_ẢNH","MÀN_HÌNH",
    "GIÁ_CẢ","TÍNH_NĂNG","THIẾT_KẾ","DỊCH_VỤ&PHỤ_KIỆN","LƯU TRỮ"
}
SENTIMENT = {"TÍCH_CỰC","TIÊU_CỰC","TRUNG_LẬP"}
QUAD_RE = re.compile(
    r"\[\s*'([^']*)'\s*,\s*'([^']*)'\s*,\s*'([^']*)'\s*,\s*'([^']*)'\s*\]"
)

def extract_json_safe(raw_text):
    """
    Robust parser for ViT5 / seq2seq ABSA output.
    """
    if not raw_text:
        return []

    s = str(raw_text)

    results = []
    for a, c, se, o in QUAD_RE.findall(s):
        a, c, se, o = a.strip(), c.strip(), se.strip(), o.strip()

        # Hard validation (quan trọng)
        if c not in CATEGORIES:
            continue
        if se not in SENTIMENT:
            continue

        results.append([a, c, se, o])

    return results


@torch.no_grad()
def get_absa_prediction_debug(reviews):
    results = [None] * len(reviews)
    uncached_reviews = []
    uncached_indices = []

    # --- 1. Check cache ---
    for i, review in enumerate(reviews):
        if review in GEN_CACHE:
            results[i] = GEN_CACHE[review]
        else:
            uncached_reviews.append(review)
            uncached_indices.append(i)

    if len(uncached_reviews) == 0:
        return results

    # --- 2. Tokenize (Seq2Seq không cần system prompt) ---
    inputs = eval_tokenizer(
        uncached_reviews,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256,
    ).to(device)

    # --- 3. Batch generate ---
    outputs = eval_model.generate(
        **inputs,
        max_new_tokens=256,
    )

    decoded = eval_tokenizer.batch_decode(outputs, skip_special_tokens=True)

    # --- 4. Parse + map lại đúng vị trí ---
    for i, out_idx in enumerate(uncached_indices):
        clean_text = decoded[i].strip()
        parsed = extract_json_safe(clean_text)

        results[out_idx] = (parsed, clean_text)
        GEN_CACHE[uncached_reviews[i]] = (parsed, clean_text)

    return results


def calculate_f1(pred, gold):
    if not pred and not gold: return 1.0
    if not pred or not gold: return 0.0
    def clean(s): return str(s).strip().lower().replace("_", " ").replace("-", " ")
    try:
        pred_set = set([tuple(clean(item) for item in x) for x in pred if isinstance(x, list) and len(x) == 4])
        gold_set = set([tuple(clean(item) for item in x) for x in gold if isinstance(x, list) and len(x) == 4])
        tp = len(pred_set & gold_set)
        if tp == 0: return 0.0
        p = tp / len(pred_set); r = tp / len(gold_set)
        return 2 * (p * r) / (p + r)
    except Exception as e: 
        print("F1 error:", e)
        return 0.0


def text_similarity(s1, s2):
    """Tính độ tương đồng giữa hai chuỗi (tỉ lệ từ 0.0 đến 1.0)"""
    return SequenceMatcher(None, s1, s2).ratio()


# --- 3. MAIN PROCESS VỚI LOGIC P- ---
def process_full_logic(input_file, output_file, sample_size=None, GLOBAL_BATCH=16):
    results = []
    count_original_chosen = 0 
    parse_miss = 0

    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            if sample_size: lines = lines[:sample_size]
    except Exception as e:
        print(f"❌ Lỗi file: {e}")
        return

    print(f"🚀 Bắt đầu xử lý {len(lines)} mẫu | GLOBAL_BATCH={GLOBAL_BATCH}")

    for batch_start in tqdm(range(0, len(lines), GLOBAL_BATCH)):
        batch_lines = lines[batch_start: batch_start + GLOBAL_BATCH]
        all_texts = []
        metas = []  # (data, gold, start_idx, n_texts)

        # Gom batch
        for line in batch_lines:
            data = json.loads(line)
            gold = ast.literal_eval(data['gold_label'])
    
            original = data["original"]
            variations = data["variations"]
    
            texts = [original] + variations
            start_idx = len(all_texts)
    
            all_texts.extend(texts)
            metas.append((data, gold, original, variations, start_idx, len(texts)))

        preds = get_absa_prediction_debug(all_texts)

        for data, gold, original, variations, start_idx, n_texts in metas:
            sample_preds = preds[start_idx: start_idx + n_texts]
            
            (pred_s, raw_s) = sample_preds[0]
            if len(pred_s) == 0: parse_miss += 1
            f1_s = calculate_f1(pred_s, gold)
            
            var_preds = sample_preds[1:]
            f1_vars = []
            for pred_v, _ in var_preds:
                if len(pred_v) == 0: parse_miss += 1
                f1_vars.append(calculate_f1(pred_v, gold))
            
            # --- 2. Similarity ---
            sim_vars = [text_similarity(original, v) for v in variations]

            # --- 3. P− selection ---
            p_minus = []
            is_simple = (len(gold) == 1)

            if is_simple:
                # CASE A: CÂU ĐƠN (Simple sentence) 
                # Chọn biến thể (s') có độ tương đồng CAO NHẤT với câu gốc [cite: 260]
                if sim_vars:
                    max_sim_idx = sim_vars.index(max(sim_vars))
                    p_minus = [variations[max_sim_idx]]
            else:
                # CASE B: CÂU PHỨC (Compound sentence) [cite: 99]
                max_f1_v = max(f1_vars) if f1_vars else 0
                
                # 1. Nếu câu gốc (s) có F1 thấp hơn biến thể tốt nhất [cite: 261]
                if f1_s < max_f1_v:
                    # Để phục vụ DPO, ta chọn biến thể có F1 thấp nhất để làm mẫu đối chứng tệ
                    min_f1_idx = f1_vars.index(min(f1_vars))
                    p_minus = [variations[min_f1_idx]]
                
                # 2. Nếu điểm F1 bằng nhau giữa câu gốc và biến thể [cite: 262]
                elif any(f == f1_s for f in f1_vars):
                    # Chọn biến thể (s') có độ tương đồng THẤP NHẤT so với câu gốc [cite: 262]
                    eq_f1_indices = [i for i, f in enumerate(f1_vars) if f == f1_s]
                    target_idx = min(eq_f1_indices, key=lambda i: sim_vars[i])
                    p_minus = [variations[target_idx]]
                
                # 3. Nếu câu gốc có điểm F1 cao hơn hẳn các biến thể [cite: 263]
                else:
                    # Chọn s' có độ tương đồng CAO NHẤT với s trong số những câu có F1 thấp hơn [cite: 263]
                    lower_f1_indices = [i for i, f in enumerate(f1_vars) if f < f1_s]
                    if lower_f1_indices:
                        target_idx = max(lower_f1_indices, key=lambda i: sim_vars[i])
                        p_minus = [variations[target_idx]]
                    # Fallback: Nếu không có câu nào thấp điểm hơn, chọn câu giống nhất
                    elif sim_vars:
                            p_minus = [variations[sim_vars.index(max(sim_vars))]]

            # Đảm bảo p_minus không bị rỗng để tránh lỗi khi huấn luyện DPO
            if not p_minus and variations:
                p_minus = [variations[0]]

            data['p_minus'] = p_minus
            data['f1_original'] = f1_s
            data['f1_variations'] = f1_vars
            
            results.append(data)

        if batch_start % 10 == 0 and batch_start != 0: print(f"Parse miss: {parse_miss}")


    # --- Write output ---
    with open(output_file, 'w', encoding='utf-8') as f:
        for entry in results:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')

    print(f"✅ Đã lưu kết quả vào {output_file}")
    print("=" * 40)
    print(f"📊 TỔNG KẾT:")
    print(f"   - Tổng số mẫu: {len(results)}")
    print(f"   - Tổng parse miss: {parse_miss}")
    print("=" * 40)

In [8]:
# --- EXECUTION ---
INPUT_PATH = '/kaggle/input/variation-atoss/vit5_atoss_results-5.jsonl'
OUTPUT_PATH = 'pm_val.jsonl'
process_full_logic(INPUT_PATH, OUTPUT_PATH, sample_size=None)

🚀 Bắt đầu xử lý 300 mẫu | GLOBAL_BATCH=16


 32%|███▏      | 6/19 [00:57<02:09,  9.98s/it]

Parse miss: 1


 58%|█████▊    | 11/19 [01:46<01:20, 10.05s/it]

Parse miss: 1


 84%|████████▍ | 16/19 [02:34<00:29,  9.90s/it]

Parse miss: 1


100%|██████████| 19/19 [02:57<00:00,  9.35s/it]

✅ Đã lưu kết quả vào pm_val.jsonl
📊 TỔNG KẾT:
   - Tổng số mẫu: 300
   - Tổng parse miss: 1
